# Tables

Builds Tables 2, 3, S1 and S2 from the data in `data/generated/`. Each is shown as a
pandas DataFrame, and its LaTeX `tabular`, laid out as in the paper, is printed and
written to `tables/`. Table 1 lists the model parameters and has no data. Run with
`make tables`.

In [1]:
import pandas as pd
from IPython.display import display

from competitive_exploration.paths import TABLES_DIR


def row(*cells):
    return " & ".join(cells) + r" \\"


def grouped_rows(table, headings, multirow=False):
    r"""Body lines for a table of formatted cells indexed by (group, row label).

    Each group starts with its heading line, or with headings[group] as a \multirow
    first column; groups are separated by \addlinespace.
    """
    lines = []
    for i, (group, part) in enumerate(table.groupby(level=0, sort=False)):
        if i:
            lines.append(r"\addlinespace")
        if not multirow:
            lines.append(headings[group])
        for j, (label, cells) in enumerate(part.droplevel(0).iterrows()):
            label = str(label).replace("%", r"\%")
            if multirow:
                first = rf"\multirow{{{len(part)}}}{{*}}{{{headings[group]}}}" if j == 0 else ""
                lines.append(row(first, label, *cells))
            else:
                lines.append(row(label, *cells))
    return lines


def write_tabular(name, align, header, body, booktabs=True):
    r"""Print a LaTeX tabular and save it as tables/<name>.tex.

    With booktabs, the table has top, middle and bottom rules, as in Tables 2, 3 and
    S2; otherwise a single \hline below the header, as in Table S1.
    """
    if booktabs:
        lines = [r"\toprule", *header, r"\midrule", *body, r"\bottomrule"]
    else:
        lines = [*header, r" \hline", *body]
    latex = "\n".join([rf"\begin{{tabular}}{{{align}}}", *lines, r"\end{tabular}"])
    print(latex)
    TABLES_DIR.mkdir(exist_ok=True)
    (TABLES_DIR / f"{name}.tex").write_text(latex + "\n")

## Table 2
Locally stable points at which both agents explore, anti-coordination points, and the
point at which both stay, cross-tabulated by whether they are Nash equilibria and whether
they are the socially optimal locally stable point, as percentages of each kind of point;
$n$ is the number of points of each kind. Pooled over the sweep of Figure S6 (Figure 5 and
the shortest horizon): $d = 2$–30, $R = 2.1$–20, $\gamma = 1 - 1/1000$, $1 - 1/5000$ and
$1 - 1/100000$.

In [2]:
import numpy as np

from competitive_exploration.paths import DATA_DIR

data = np.load(DATA_DIR / "sweep_nash_social_dilemma_fine401.npz")
# The sweep of Figure S6: every discount factor, d = 2-30.
a_keep = data["d_values"] >= 2
parts, n = {}, {}
for label, key in [
    ("Both explore", "symmetric_counts"),
    ("Anti-coordination", "anti_coordination_counts"),
    ("Both stay", "both_stay_counts"),
]:
    counts = data[key][:, a_keep].sum(axis=(0, 1, 2))
    n[label] = counts.sum()
    parts[label] = pd.DataFrame(
        100 * counts / counts.sum(),
        index=["Nash", "Not Nash"],
        columns=["Socially optimal (%)", "Not socially optimal (%)"],
    )
table2 = pd.concat(parts, names=["Locally stable point", ""])
display(table2.round(1))
print("n:", n)

write_tabular(
    "table2",
    "@{}llrr@{}",
    [row("", "", "Socially", "Not socially"), row("Locally stable point", "", "optimal", "optimal")],
    grouped_rows(
        table2.map(lambda p: rf"{p:.1f}\%"),
        {
            "Both explore": rf"Both explore, $(\eps,\eps)$, $n = {n['Both explore']}$",
            "Anti-coordination": rf"Anti-coordination, $(\eps,0)$, $n = {n['Anti-coordination']}$",
            "Both stay": rf"Both stay, $(0,0)$, $n = {n['Both stay']}$",
        },
        multirow=True,
    ),
)

Socially optimal (%)  Not socially optimal (%)
Locally stable point                                                         
Both explore         Nash                      51.5                       5.9
                     Not Nash                   0.0                      42.6
Anti-coordination    Nash                      20.2                       0.0
                     Not Nash                  29.9                      49.9
Both stay            Nash                       0.2                      24.6
                     Not Nash                   0.0                      75.2

n: {'Both explore': np.int64(17604), 'Anti-coordination': np.int64(34726), 'Both stay': np.int64(17400)}
\begin{tabular}{@{}llrr@{}}
\toprule
 &  & Socially & Not socially \\
Locally stable point &  & optimal & optimal \\
\midrule
\multirow{2}{*}{Both explore, $(\eps,\eps)$, $n = 17604$} & Nash & 51.5\% & 5.9\% \\
 & Not Nash & 0.0\% & 42.6\% \\
\addlinespace
\multirow{2}{*}{Anti-coordination, $(\eps,0)$, $n = 34726$} & Nash & 20.2\% & 0.0\% \\
 & Not Nash & 29.9\% & 49.9\% \\
\addlinespace
\multirow{2}{*}{Both stay, $(0,0)$, $n = 17400$} & Nash & 0.2\% & 24.6\% \\
 & Not Nash & 0.0\% & 75.2\% \\
\bottomrule
\end{tabular}


## Table 3
Locally stable points without and with signaling at $d = 4$, $R = 2.1$,
$\gamma = 1 - 1/5000$, on a grid of 51 exploration rates. Of two points related by
exchanging the agents, one is listed and marked *. The last signaling row is the
baseline's symmetric Nash equilibrium, which is not locally stable with signaling.
$\pi_1$ and $\pi_2$ are the payoffs to the agents with the first and second strategy.

In [3]:
import numpy as np

from competitive_exploration.paths import DATA_DIR

data = np.load(DATA_DIR / "locally_stable_points.npz")


def strategy(eps):
    """One agent's strategy: a rate, or a (rate, post-signal rate) pair."""
    eps = [round(float(e), 2) for e in np.atleast_1d(eps)]
    return str(eps[0]) if len(eps) == 1 else f"({eps[0]}, {eps[1]})"


def stable_points(pairs, payoff, payoffs, nash, locally_stable):
    best = payoff[locally_stable].max()
    rows = {}
    for k in np.lexsort((-payoff, ~locally_stable)):
        if tuple(np.atleast_1d(pairs[k][0])) < tuple(np.atleast_1d(pairs[k][1])):
            continue  # its counterpart with the agents exchanged is listed
        first, second = strategy(pairs[k][0]), strategy(pairs[k][1])
        rows[f"({first}, {second})" + ("*" if first != second else "")] = {
            "Payoff": round(float(payoff[k])),
            "pi_1": round(float(payoffs[k][0])),
            "pi_2": round(float(payoffs[k][1])),
            "Nash": bool(nash[k]),
            "Social optimum": bool(locally_stable[k] and np.isclose(payoff[k], best)),
            "Locally stable": bool(locally_stable[k]),
        }
    return pd.DataFrame.from_dict(rows, orient="index")


baseline_payoff = data["baseline_payoff"]
table3 = pd.concat(
    {
        "No signaling": stable_points(
            data["baseline_eps"],
            baseline_payoff,
            data["baseline_payoffs"],
            data["baseline_nash"],
            np.ones(len(baseline_payoff), dtype=bool),
        ),
        "Signaling": stable_points(
            data["signaling_eps"].reshape(-1, 2, 2),
            data["signaling_payoff"],
            data["signaling_payoffs"],
            data["signaling_nash"],
            data["signaling_locally_stable"],
        ),
    },
    names=["Model", "Strategy pair"],
)
display(table3)

write_tabular(
    "table3",
    "@{}lrrrccc@{}",
    [row("Strategy pair", "Payoff", r"$\pi_1$", r"$\pi_2$", "Nash", "Social opt.", r"Loc.\ stable")],
    grouped_rows(
        table3.map(lambda v: ("Yes" if v else "No") if isinstance(v, bool) else f"{v:.0f}"),
        {
            "No signaling": r"\multicolumn{7}{@{}l}{\textit{No signaling}, $(\eps_1, \eps_2)$} \\",
            "Signaling": r"\multicolumn{7}{@{}l}{\textit{Signaling}, "
            r"$((\eps_1, \eps_1'), (\eps_2, \eps_2'))$} \\",
        },
    ),
)

Payoff  pi_1  pi_2   Nash  \
Model        Strategy pair                                             
No signaling (0.3, 0.0)*                    11082  6389  4693   True   
             (0.28, 0.28)                    9357  4678  4678   True   
             (0.0, 0.0)                      5000  2500  2500  False   
Signaling    ((0.36, 0.0), (0.36, 0.0))     11362  5681  5681   True   
             ((0.3, 0.3), (0.0, 0.0))*      11082  6389  4693  False   
             ((0.36, 0.22), (0.3, 0.0))*    10849  5816  5033  False   
             ((0.3, 0.22), (0.3, 0.22))      9856  4928  4928  False   
             ((0.3, 0.3), (0.0, 0.22))*      9477  4998  4479  False   
             ((0.0, 0.0), (0.0, 0.0))        5000  2500  2500  False   
             ((0.28, 0.28), (0.28, 0.28))    9357  4678  4678  False   

                                           Social optimum  Locally stable  
Model        Strategy pair                                                 
No signaling (0.3, 0.0)*                             True            True  
             (0.28, 0.28)                           False            True  
             (0.0, 0.0)                             False            True  
Signaling    ((0.36, 0.0), (0.36, 0.0))              True            True  
             ((0.3, 0.3), (0.0, 0.0))*              False            True  
             ((0.36, 0.22), (0.3, 0.0))*            False            True  
             ((0.3, 0.22), (0.3, 0.22))             False            True  
             ((0.3, 0.3), (0.0, 0.22))*             False            True  
             ((0.0, 0.0), (0.0, 0.0))               False            True  
             ((0.28, 0.28), (0.28, 0.28))           False           False

\begin{tabular}{@{}lrrrccc@{}}
\toprule
Strategy pair & Payoff & $\pi_1$ & $\pi_2$ & Nash & Social opt. & Loc.\ stable \\
\midrule
\multicolumn{7}{@{}l}{\textit{No signaling}, $(\eps_1, \eps_2)$} \\
(0.3, 0.0)* & 11082 & 6389 & 4693 & Yes & Yes & Yes \\
(0.28, 0.28) & 9357 & 4678 & 4678 & Yes & No & Yes \\
(0.0, 0.0) & 5000 & 2500 & 2500 & No & No & Yes \\
\addlinespace
\multicolumn{7}{@{}l}{\textit{Signaling}, $((\eps_1, \eps_1'), (\eps_2, \eps_2'))$} \\
((0.36, 0.0), (0.36, 0.0)) & 11362 & 5681 & 5681 & Yes & Yes & Yes \\
((0.3, 0.3), (0.0, 0.0))* & 11082 & 6389 & 4693 & No & No & Yes \\
((0.36, 0.22), (0.3, 0.0))* & 10849 & 5816 & 5033 & No & No & Yes \\
((0.3, 0.22), (0.3, 0.22)) & 9856 & 4928 & 4928 & No & No & Yes \\
((0.3, 0.3), (0.0, 0.22))* & 9477 & 4998 & 4479 & No & No & Yes \\
((0.0, 0.0), (0.0, 0.0)) & 5000 & 2500 & 2500 & No & No & Yes \\
((0.28, 0.28), (0.28, 0.28)) & 9357 & 4678 & 4678 & No & No & No \\
\bottomrule
\end{tabular}


## Table S1
Percentage of parameter combinations ($d = 2$–30, $R = 2.1$–20) with a dilemma of each
kind, judged by collective payoff and by Pareto dominance. The paper reports the seven
longest foraging horizons.

In [4]:
import numpy as np

from competitive_exploration.paths import DATA_DIR

data = np.load(DATA_DIR / "dilemma_kinds.npz")
kinds = ["first", "first_pareto", "second", "second_pareto", "strict", "strict_pareto"]
shown = [(t, T) for t, T in enumerate(data["T_values"]) if T >= 1000]
table_s1 = pd.DataFrame(
    [[100 * data[kind][t].mean() for kind in kinds] for t, _ in shown],
    index=pd.Index([f"1 - 1/{T}" for _, T in shown], name="gamma"),
    columns=pd.MultiIndex.from_product(
        [["First kind", "Second kind", "Strict"], ["Collective (%)", "Pareto (%)"]]
    ),
)
display(table_s1.round(1))

write_tabular(
    "table_s1",
    "l|rr|rr|rr",
    [
        r" & \multicolumn{2}{c|}{First kind} & \multicolumn{2}{c|}{Second kind}"
        r" & \multicolumn{2}{c}{Strict} \\",
        row(r"$\gamma$", *[r"Collective (\%)", r"Pareto (\%)"] * 3),
    ],
    [row(rf"$1-1/{T}$", *(f"{v:.1f}" for v in values)) for (_, T), values in zip(shown, table_s1.values)],
    booktabs=False,
)

First kind               Second kind             \
              Collective (%) Pareto (%) Collective (%) Pareto (%)   
gamma                                                               
1 - 1/1000             100.0      100.0           99.2       92.3   
1 - 1/5000             100.0      100.0          100.0       98.2   
1 - 1/10000            100.0      100.0          100.0       99.3   
1 - 1/50000            100.0      100.0          100.0      100.0   
1 - 1/100000           100.0      100.0          100.0      100.0   
1 - 1/500000           100.0      100.0          100.0      100.0   
1 - 1/1000000          100.0      100.0          100.0      100.0   

                      Strict             
              Collective (%) Pareto (%)  
gamma                                    
1 - 1/1000              48.1        0.9  
1 - 1/5000              26.8        0.4  
1 - 1/10000             21.2        0.1  
1 - 1/50000             16.0        0.2  
1 - 1/100000            15.2        0.1  
1 - 1/500000            14.2        0.1  
1 - 1/1000000           14.2        0.1

\begin{tabular}{l|rr|rr|rr}
 & \multicolumn{2}{c|}{First kind} & \multicolumn{2}{c|}{Second kind} & \multicolumn{2}{c}{Strict} \\
$\gamma$ & Collective (\%) & Pareto (\%) & Collective (\%) & Pareto (\%) & Collective (\%) & Pareto (\%) \\
 \hline
$1-1/1000$ & 100.0 & 100.0 & 99.2 & 92.3 & 48.1 & 0.9 \\
$1-1/5000$ & 100.0 & 100.0 & 100.0 & 98.2 & 26.8 & 0.4 \\
$1-1/10000$ & 100.0 & 100.0 & 100.0 & 99.3 & 21.2 & 0.1 \\
$1-1/50000$ & 100.0 & 100.0 & 100.0 & 100.0 & 16.0 & 0.2 \\
$1-1/100000$ & 100.0 & 100.0 & 100.0 & 100.0 & 15.2 & 0.1 \\
$1-1/500000$ & 100.0 & 100.0 & 100.0 & 100.0 & 14.2 & 0.1 \\
$1-1/1000000$ & 100.0 & 100.0 & 100.0 & 100.0 & 14.2 & 0.1 \\
\end{tabular}


## Table S2
Strict dilemmas whose worst Nash equilibrium has both agents exploring: the percentage
that signaling removes, the number removed out of the number of dilemmas, and the mean
and largest rise in the collective payoff at the worst Nash equilibrium among those
removed, for collective-only dilemmas, Pareto dilemmas, and all of them.

In [5]:
import numpy as np

from competitive_exploration.paths import DATA_DIR

base = np.load(DATA_DIR / "sweep_nash_social_dilemma_fine401.npz")
signaling = np.load(DATA_DIR / "signaling_fine401.npz")
solved = signaling["status"] == 0
improvement = (
    100 * (signaling["worst_nash_payoff"] - base["worst_nash_payoff"]) / base["worst_nash_payoff"]
)
groups = [
    signaling["candidate"] & ~base["pareto_dilemma"],
    signaling["candidate"] & base["pareto_dilemma"],
    signaling["candidate"],
]

index, values = [], []
for t, T in enumerate(base["T_values"]):
    for label, summary in [
        ("Dilemmas solved (%)", lambda g: 100 * solved[t][g[t]].mean()),
        ("Dilemmas solved (count)", lambda g: f"{solved[t][g[t]].sum()} / {g[t].sum()}"),
        ("Mean worst-Nash improvement (%)", lambda g: np.nanmean(improvement[t][g[t] & solved[t]])),
        ("Largest worst-Nash improvement (%)", lambda g: np.nanmax(improvement[t][g[t] & solved[t]])),
    ]:
        index.append((f"1 - 1/{T}", label))
        values.append([summary(g) for g in groups])
table_s2 = pd.DataFrame(
    values,
    index=pd.MultiIndex.from_tuples(index, names=["gamma", ""]),
    columns=["Collective only", "Pareto", "All"],
)
display(table_s2.map(lambda v: v if isinstance(v, str) else round(v, 1)))

write_tabular(
    "table_s2",
    "lrrr",
    [row("", r"\textbf{Collective only}", r"\textbf{Pareto}", r"\textbf{All}")],
    grouped_rows(
        table_s2.map(lambda v: v if isinstance(v, str) else f"{v:.1f}"),
        {f"1 - 1/{T}": rf"\multicolumn{{4}}{{l}}{{$\gamma = 1-1/{T}$}} \\" for T in base["T_values"]},
    ),
)

Collective only  Pareto  \
gamma                                                                     
1 - 1/1000   Dilemmas solved (%)                           28.7    83.3   
             Dilemmas solved (count)                   35 / 122   5 / 6   
             Mean worst-Nash improvement (%)                6.0    10.2   
             Largest worst-Nash improvement (%)            14.2    19.3   
1 - 1/5000   Dilemmas solved (%)                           23.2   100.0   
             Dilemmas solved (count)                   57 / 246   6 / 6   
             Mean worst-Nash improvement (%)                4.6    12.3   
             Largest worst-Nash improvement (%)            16.4    20.9   
1 - 1/100000 Dilemmas solved (%)                            7.7    45.5   
             Dilemmas solved (count)                   41 / 534  5 / 11   
             Mean worst-Nash improvement (%)                4.5     8.3   
             Largest worst-Nash improvement (%)            11.0    15.1   

                                                      All  
gamma                                                      
1 - 1/1000   Dilemmas solved (%)                     31.2  
             Dilemmas solved (count)             40 / 128  
             Mean worst-Nash improvement (%)          6.5  
             Largest worst-Nash improvement (%)      19.3  
1 - 1/5000   Dilemmas solved (%)                     25.0  
             Dilemmas solved (count)             63 / 252  
             Mean worst-Nash improvement (%)          5.4  
             Largest worst-Nash improvement (%)      20.9  
1 - 1/100000 Dilemmas solved (%)                      8.4  
             Dilemmas solved (count)             46 / 545  
             Mean worst-Nash improvement (%)          4.9  
             Largest worst-Nash improvement (%)      15.1

\begin{tabular}{lrrr}
\toprule
 & \textbf{Collective only} & \textbf{Pareto} & \textbf{All} \\
\midrule
\multicolumn{4}{l}{$\gamma = 1-1/1000$} \\
Dilemmas solved (\%) & 28.7 & 83.3 & 31.2 \\
Dilemmas solved (count) & 35 / 122 & 5 / 6 & 40 / 128 \\
Mean worst-Nash improvement (\%) & 6.0 & 10.2 & 6.5 \\
Largest worst-Nash improvement (\%) & 14.2 & 19.3 & 19.3 \\
\addlinespace
\multicolumn{4}{l}{$\gamma = 1-1/5000$} \\
Dilemmas solved (\%) & 23.2 & 100.0 & 25.0 \\
Dilemmas solved (count) & 57 / 246 & 6 / 6 & 63 / 252 \\
Mean worst-Nash improvement (\%) & 4.6 & 12.3 & 5.4 \\
Largest worst-Nash improvement (\%) & 16.4 & 20.9 & 20.9 \\
\addlinespace
\multicolumn{4}{l}{$\gamma = 1-1/100000$} \\
Dilemmas solved (\%) & 7.7 & 45.5 & 8.4 \\
Dilemmas solved (count) & 41 / 534 & 5 / 11 & 46 / 545 \\
Mean worst-Nash improvement (\%) & 4.5 & 8.3 & 4.9 \\
Largest worst-Nash improvement (\%) & 11.0 & 15.1 & 15.1 \\
\bottomrule
\end{tabular}
